In [1]:
import json
import numpy as np
from pandas import read_csv
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
import joblib



In [2]:
#Variables generales
ruta_model = "../../Model/"
ruta_metrics = "../../Metrics/"
ruta_df = "../../Datasets/"

semilla = 111

## FUNCIÓN BASE

In [3]:
def train_random_forest(X, y, output_model_path='best_model.pkl', output_metrics_path='metrics.json', kf = 2, param_grid = {'n_estimators' : [100]}):    
    # Random Forest Classifier
    rf = RandomForestClassifier(random_state = 42)
    
    # Definir validación cruzada de K-Fold
    kf = KFold(n_splits = kf
               , shuffle = True
               , random_state = 42
               )
    
    # Cross Validation
    grid_search = GridSearchCV(estimator = rf
                               , param_grid = param_grid
                               , cv = kf
                               , scoring = 'accuracy'
                               , n_jobs = -1
                               , verbose = 2
                               )
    
    # Ajustar el modelo
    grid_search.fit(X, y)
    
    # mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guarde el mejor modelo en un archivo .pkl
    joblib.dump(best_model, output_model_path)
    
    # mejor modelo.
    y_pred = best_model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average = 'weighted'),
        'recall': recall_score(y, y_pred, average='weighted'),
        'f1_score': f1_score(y, y_pred, average='weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class='ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict = True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guarde las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)

    return metrics, best_model

## ENTRENAMIENTO

### df interpolation

* SMOTE

In [4]:
# carga de caracteristicas
df_IM_smote = read_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [5]:
%%time
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

metric_1, model_1 = train_random_forest( X = df_IM_smote.drop(columns='FLAG')
                                        , y = df_IM_smote['FLAG']
                                        , output_model_path = '{}RF_IM_S.pkl'.format(ruta_model)
                                        , output_metrics_path = '{}RF_IM_S.json'.format(ruta_metrics)
                                        , kf = 3
                                       , param_grid = param_grid)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Wall time: 7min 42s


In [6]:
metric_1

{'accuracy': 0.9948703276041773,
 'precision': 0.9949149003873728,
 'recall': 0.9948703276041773,
 'f1_score': 0.9948673042791466,
 'roc_auc': 0.9942428845834322,
 'confusion_matrix': [[27181, 3], [248, 21499]],
 'classification_report': {'0': {'precision': 0.9909584746071676,
   'recall': 0.9998896409652737,
   'f1-score': 0.9954040246827679,
   'support': 27184},
  '1': {'precision': 0.999860478095061,
   'recall': 0.988596128201591,
   'f1-score': 0.994196397604569,
   'support': 21747},
  'accuracy': 0.9948703276041773,
  'macro avg': {'precision': 0.9954094763511143,
   'recall': 0.9942428845834324,
   'f1-score': 0.9948002111436685,
   'support': 48931},
  'weighted avg': {'precision': 0.9949149003873728,
   'recall': 0.9948703276041773,
   'f1-score': 0.9948673042791466,
   'support': 48931}},
 'balanced_accuracy': 0.9942428845834324,
 'cohen_kappa': 0.9896006865676011,
 'matthews_corrcoef': 0.9896516733512705}

* ADASYN

In [7]:
# carga de caracteristicas
df_IM_adasyn = read_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [8]:
%%time
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

metric_2, model_2 = train_random_forest( X = df_IM_adasyn.drop(columns='FLAG')
                                        , y = df_IM_adasyn['FLAG']
                                        , output_model_path = '{}RF_IM_A.pkl'.format(ruta_model)
                                        , output_metrics_path = '{}RF_IM_A.json'.format(ruta_metrics)
                                        , kf = 3
                                       , param_grid = param_grid)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Wall time: 7min 30s


In [9]:
metric_2

{'accuracy': 0.9944127218026647,
 'precision': 0.9944665536987364,
 'recall': 0.9944127218026647,
 'f1_score': 0.9944090340072662,
 'roc_auc': 0.993712348683001,
 'confusion_matrix': [[27182, 2], [271, 21406]],
 'classification_report': {'0': {'precision': 0.9901285833970787,
   'recall': 0.9999264273101824,
   'f1-score': 0.9950033859838571,
   'support': 27184},
  '1': {'precision': 0.9999065769805681,
   'recall': 0.9874982700558196,
   'f1-score': 0.9936636880584891,
   'support': 21677},
  'accuracy': 0.9944127218026647,
  'macro avg': {'precision': 0.9950175801888234,
   'recall': 0.9937123486830011,
   'f1-score': 0.9943335370211731,
   'support': 48861},
  'weighted avg': {'precision': 0.9944665536987364,
   'recall': 0.9944127218026647,
   'f1-score': 0.9944090340072662,
   'support': 48861}},
 'balanced_accuracy': 0.9937123486830011,
 'cohen_kappa': 0.9886674223959592,
 'matthews_corrcoef': 0.9887290673473695}

### df linear regression

* SMOTE

In [10]:
# carga de caracteristicas
df_LR_smote = read_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [11]:
%%time
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

metric_3, model_3 = train_random_forest( X = df_LR_smote.drop(columns='FLAG')
                                        , y = df_LR_smote['FLAG']
                                        , output_model_path = '{}RF_LR_S.pkl'.format(ruta_model)
                                        , output_metrics_path = '{}RF_LR_S.json'.format(ruta_metrics)
                                        , kf = 3
                                       , param_grid = param_grid)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Wall time: 7min 10s


In [12]:
metric_3

{'accuracy': 0.9980789274692935,
 'precision': 0.9980838720342583,
 'recall': 0.9980789274692935,
 'f1_score': 0.9980785596704748,
 'roc_auc': 0.9978663733881367,
 'confusion_matrix': [[27178, 6], [88, 21659]],
 'classification_report': {'0': {'precision': 0.996772537225849,
   'recall': 0.9997792819305473,
   'f1-score': 0.9982736455463728,
   'support': 27184},
  '1': {'precision': 0.999723055619663,
   'recall': 0.9959534648457259,
   'f1-score': 0.9978347000829263,
   'support': 21747},
  'accuracy': 0.9980789274692935,
  'macro avg': {'precision': 0.9982477964227561,
   'recall': 0.9978663733881366,
   'f1-score': 0.9980541728146495,
   'support': 48931},
  'weighted avg': {'precision': 0.9980838720342583,
   'recall': 0.9980789274692935,
   'f1-score': 0.9980785596704748,
   'support': 48931}},
 'balanced_accuracy': 0.9978663733881366,
 'cohen_kappa': 0.9961083566994376,
 'matthews_corrcoef': 0.9961140967853596}

* ADASYN

In [13]:
# carga de caracteristicas
df_LR_adasyn = read_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [14]:
%%time
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

metric_4, model_4 = train_random_forest( X = df_LR_adasyn.drop(columns='FLAG')
                                        , y = df_LR_adasyn['FLAG']
                                        , output_model_path = '{}RF_LR_A.pkl'.format(ruta_model)
                                        , output_metrics_path = '{}RF_LR_A.json'.format(ruta_metrics)
                                        , kf = 3
                                       , param_grid = param_grid)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Wall time: 6min 30s


In [15]:
metric_4

{'accuracy': 0.9992605525315805,
 'precision': 0.9992612136517253,
 'recall': 0.9992605525315805,
 'f1_score': 0.999260498315976,
 'roc_auc': 0.9991774143154818,
 'confusion_matrix': [[27181, 3], [33, 21468]],
 'classification_report': {'0': {'precision': 0.9987873888439773,
   'recall': 0.9998896409652737,
   'f1-score': 0.9993382109636384,
   'support': 27184},
  '1': {'precision': 0.9998602766522285,
   'recall': 0.9984651876656899,
   'f1-score': 0.9991622451829097,
   'support': 21501},
  'accuracy': 0.9992605525315805,
  'macro avg': {'precision': 0.9993238327481029,
   'recall': 0.9991774143154818,
   'f1-score': 0.999250228073274,
   'support': 48685},
  'weighted avg': {'precision': 0.9992612136517253,
   'recall': 0.9992605525315805,
   'f1-score': 0.999260498315976,
   'support': 48685}},
 'balanced_accuracy': 0.9991774143154818,
 'cohen_kappa': 0.99850045672389,
 'matthews_corrcoef': 0.9985012363283164}